# 05 — Dataset comparison, hyperparameter tuning and walk-forward validation

Runs on a **GPU runtime** (`Runtime` -> `Change runtime type` -> T4).

1. Train the baseline autoencoder (`src.train`) on the original v9 and on v9_1 and compare.
2. Random-search hyperparameters on v9_1, scored by walk-forward validation (folds A-C).
3. Walk-forward results for baseline vs tuned on all four folds; fold D (2024-2025) was never seen by the search.

Results are pushed to `colab-training` under `reports/tuning/`. The search is resumable: if Colab disconnects, re-run the cells and finished trials are skipped.

In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO = "DaniellaWambua/Real-Time-Flood-Data-Pipeline"
BRANCH = "colab-training"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -q -r requirements-colab.txt

In [ ]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 1. Baseline on both datasets

In [ ]:
!mkdir -p reports/tuning
!FLOOD_DATASET=v9_orig python -m src.train 2>&1 | tee reports/tuning/baseline_train_v9_orig.log | grep -A9 "^model "
!FLOOD_DATASET=v9_1 python -m src.train 2>&1 | tee reports/tuning/baseline_train_v9_1.log | grep -A9 "^model "

## 2. Hyperparameter search (v9_1, folds A-C)

`--trials 20` is trial 0 (the baseline) plus 19 random configurations, each trained on 6 catchments x 3 folds. Raise it if you have the time. Window length, encoder size, learning rate, batch size and dropout are searched; see `SEARCH_SPACE` in `src/tune_autoencoder.py`.

In [ ]:
!FLOOD_DATASET=v9_1 python -m src.tune_autoencoder --stage search --trials 20 2>&1 | grep -v -i "warn\|cuda\|tensorflow/"

## 3. Walk-forward: baseline vs tuned, folds A-D

In [ ]:
!FLOOD_DATASET=v9_1 python -m src.tune_autoencoder --stage final 2>&1 | grep -v -i "warn\|cuda\|tensorflow/"

## Push results back

In [ ]:
# models/ is left untouched: baseline re-training above overwrote it locally on this VM only.
!git checkout -- models/ reports/figures/
!git add reports/tuning/
!git -c user.email="youngkyper254@gmail.com" -c user.name="Daniella Wambua" \
     commit -m "Colab: v9 vs v9_1 baseline, hyperparameter search and walk-forward results"
!git push origin colab-training